In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

DIRETORIO_BASE = "."
# Dataset original (nao embaralhado): selecao mexe so em COLUNAS, nao em linhas.
CAMINHO_ARQUIVO = f"{DIRETORIO_BASE}/dados/mh1m_balanceadas.npz'

dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)
X = dados['data']
y = dados['classes']
colunas = dados['column_names']
print(f"Dataset original: X={X.shape}, y={y.shape}, colunas={colunas.shape}")

In [ ]:
# --- Parametros gerais ---
FATOR = 1  # limiar = FATOR * (1/n_features_do_grupo); 1 = acima da media uniforme (mais permissivo)

BASE_SRC_SHAP = f"{DIRETORIO_BASE}/src_shap"
BASE_DADOS = f"{DIRETORIO_BASE}/dados"

# Novo

In [ ]:
# --- Funcoes auxiliares ---

def seleciona_features_de_grupo(nome_grupo, fator=FATOR):
    """Le o ranking medio normalizado de um grupo, aplica o corte
    limiar = fator * (1/n_features) e retorna (nomes_selecionados, registro_orcamento)."""
    caminho_csv = f"{BASE_SRC_SHAP}/{nome_grupo}/features_importancias_medias_geral.csv"
    df = pd.read_csv(caminho_csv)

    n_features = len(df)                     # universo do grupo (pos-merge dos 3 modelos)
    limiar = fator * (1.0 / n_features)

    selecionadas = df[df["importance_media"] >= limiar]
    nomes = selecionadas["feature"].tolist()

    registro = {
        "grupo": nome_grupo,
        "n_features_total": n_features,
        "n_features_selecionadas": len(nomes),
        "limiar": limiar,
        "fator": fator,
    }
    print(f"[{nome_grupo}] total={n_features} | limiar={limiar:.3e} | selecionadas={len(nomes)}")
    return nomes, registro


def conta_por_namespace(nomes):
    """Conta quantas features selecionadas pertencem a cada namespace (para o orcamento por familia)."""
    contagem = {"intents": 0, "permissions": 0, "opcodes": 0, "apicalls": 0}
    for nome in nomes:
        if nome.startswith("intents::"):
            contagem["intents"] += 1
        elif nome.startswith("permissions::"):
            contagem["permissions"] += 1
        elif nome.startswith("opcodes::"):
            contagem["opcodes"] += 1
        elif nome.startswith("apicalls::"):
            contagem["apicalls"] += 1
    return contagem


def monta_e_salva_dataset(nomes_features, caminho_saida, rotulo):
    """Aplica a mascara sobre o dataset original e salva o .npz reduzido."""
    nomes_arr = np.array(list(dict.fromkeys(nomes_features)))  # unicidade preservando ordem

    existe = np.isin(nomes_arr, colunas)
    if not existe.all():
        faltando = nomes_arr[~existe]
        print(f"  ATENCAO [{rotulo}]: {len(faltando)} feature(s) nao encontrada(s) em column_names. Ex.: {faltando[:5]}")

    mask = np.isin(colunas, nomes_arr)
    X_red = X[:, mask]
    colunas_red = colunas[mask]
    print(f"  [{rotulo}] dataset reduzido: data={X_red.shape}, column_names={colunas_red.shape}")

    np.savez_compressed(caminho_saida, data=X_red, classes=y, column_names=colunas_red)
    print(f"  [{rotulo}] salvo em: {caminho_saida}")

    chk = np.load(caminho_saida, allow_pickle=True)
    print(f"  [{rotulo}] verificacao -> data={chk['data'].shape}, classes={chk['classes'].shape}, column_names={chk['column_names'].shape}")
    return len(colunas_red)

In [ ]:
# ============================================================
# DATASET 1 - Uniao das selecoes dos 4 grupos INDIVIDUAIS
# Cada grupo selecionado na sua propria regua; depois unimos as features.
# ============================================================
print("=== DATASET 1: uniao dos grupos individuais ===")
grupos_individuais = ["intents", "permissions", "opcodes", "apicalls"]

features_uniao = []
orcamento_ds1 = []
for nome_grupo in tqdm(grupos_individuais):
    nomes, registro = seleciona_features_de_grupo(nome_grupo)
    features_uniao.extend(nomes)
    orcamento_ds1.append(registro)

# Orcamento por grupo (numero por familia = proprio n_features_selecionadas de cada grupo)
df_orc1 = pd.DataFrame(orcamento_ds1)
df_orc1["origem"] = "individuais_uniao"
df_orc1.to_csv(f"{BASE_DADOS}/orcamento_shap_individuais.csv", index=False)
print(df_orc1)

monta_e_salva_dataset(
    features_uniao,
    f"{BASE_DADOS}/mh1m_balanceadas_shap.npz",
    "DS1 individuais",
)
print()